# Classifying Storm Rainfall Patterns with Machine Learning

*Geography + AI Teaching Notebooks · 1 of 3*

Storms do not just differ in *how much* rain they bring — they differ in
*how that rain is arranged in space*. A compact, intense core and a broad,
gentle shield can deliver the same total rainfall but produce very
different flood responses downstream.

This notebook is a teaching module on **unsupervised machine learning**.
It shows how to take a set of rainfall fields, reduce their dimensionality,
and let an algorithm discover recurring spatial *regimes* — all with
synthetic, reproducible data that runs on any laptop.

**What students practise here**
1. Generating and visualising gridded spatial data.
2. Reducing dimensionality with Principal Component Analysis.
3. Clustering with k-means.
4. Interpreting cluster centroids as physically meaningful patterns.
5. Connecting an abstract method to a real geographic question.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)
GRID = 32        # 32x32 grid representing a radar rainfall patch
N_EVENTS = 200   # number of synthetic storm events


## 1. Synthetic storm rainfall fields

We simulate four canonical rainfall structures. In a real course, students
would later swap in archived radar data — the workflow is identical.

- **Concentric** — a strong, roughly circular core (think of a mature
  cyclone eyewall).
- **Asymmetric** — rain concentrated on one side of the storm.
- **Distributed** — a broad shield of lower-intensity rain.
- **Multi-cell** — several discrete convective maxima.


In [ ]:
def gaussian_blob(grid, center, sigma, peak):
    y, x = np.mgrid[0:grid, 0:grid]
    r2 = (x - center[0])**2 + (y - center[1])**2
    return peak * np.exp(-r2 / (2 * sigma**2))

def make_concentric(rng):
    field = gaussian_blob(GRID, (16, 16), 4, 60)
    return field + rng.normal(0, 2, (GRID, GRID))

def make_asymmetric(rng):
    field = gaussian_blob(GRID, (10, 16), 5, 70)
    field += gaussian_blob(GRID, (22, 16), 3, 25)
    return field + rng.normal(0, 2, (GRID, GRID))

def make_distributed(rng):
    field = gaussian_blob(GRID, (12, 14), 9, 30)
    field += gaussian_blob(GRID, (22, 20), 8, 22)
    return field + rng.normal(0, 3, (GRID, GRID))

def make_multicell(rng):
    field = np.zeros((GRID, GRID))
    for _ in range(4):
        cx, cy = rng.integers(6, 26, size=2)
        field += gaussian_blob(GRID, (cx, cy), 2.5, rng.uniform(35, 55))
    return field + rng.normal(0, 2, (GRID, GRID))

GENERATORS = [make_concentric, make_asymmetric, make_distributed, make_multicell]
labels = rng.integers(0, len(GENERATORS), size=N_EVENTS)
events = np.stack([GENERATORS[k](rng) for k in labels])
events = np.clip(events, 0, None)  # rainfall cannot be negative
print(f'Generated a tensor of shape {events.shape}')


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
titles = ['Concentric', 'Asymmetric', 'Distributed', 'Multi-cell']
for col in range(4):
    for row in range(2):
        idx = np.where(labels == col)[0][row]
        ax = axes[row, col]
        ax.imshow(events[idx], cmap='Blues', vmin=0, vmax=80)
        if row == 0:
            ax.set_title(titles[col])
        ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('Eight example synthetic rainfall fields', y=1.02)
plt.tight_layout(); plt.show()


## 2. Dimensionality reduction with PCA

Each 32x32 field is 1024 numbers. PCA finds the handful of combined
patterns ('principal components') that capture most of the variation,
so that later steps work in a much smaller, less noisy space.


In [ ]:
X = events.reshape(N_EVENTS, -1)
Xs = StandardScaler().fit_transform(X)

pca = PCA(n_components=10, random_state=0)
Z = pca.fit_transform(Xs)
print('Explained variance ratio:', np.round(pca.explained_variance_ratio_, 3))


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o')
plt.xlabel('Principal component index')
plt.ylabel('Cumulative explained variance')
plt.title('How much pattern each component captures')
plt.grid(alpha=0.3); plt.show()


## 3. Clustering the storms

k-means groups the events in the reduced PCA space. In a real study we
would not know the 'true' labels — the clusters are interpreted as
*emergent* rainfall regimes that the data reveal on their own.


In [ ]:
kmeans = KMeans(n_clusters=4, n_init=20, random_state=0)
clusters = kmeans.fit_predict(Z)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(Z[:, 0], Z[:, 1], c=clusters, cmap='tab10', s=22, alpha=0.8)
ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2')
ax.set_title('k-means clusters in PCA space')
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
for k in range(4):
    mean_field = events[clusters == k].mean(axis=0)
    axes[k].imshow(mean_field, cmap='Blues', vmin=0, vmax=80)
    axes[k].set_title(f'Cluster {k}\nn={(clusters == k).sum()}')
    axes[k].set_xticks([]); axes[k].set_yticks([])
fig.suptitle('Cluster centroids — the rainfall regimes the data revealed', y=1.05)
plt.tight_layout(); plt.show()


## 4. Why this matters geographically

Spatial structure, not just total rainfall, shapes how a landscape
responds. The same clustering can become a categorical input to a
rainfall-runoff model, helping with:

- **Reservoir and flood-warning operations** — anticipating the response
  to an approaching storm of a known *type*.
- **Sediment and erosion estimates** — different storm regimes mobilise
  very different amounts of material.
- **Climate analysis** — tracking whether certain storm types are
  becoming more frequent.

### Suggested extensions for students

1. Replace the synthetic data with an open radar archive.
2. Try other clusterers (Gaussian Mixture, HDBSCAN) and compare.
3. Vary the number of clusters and discuss how to choose it.
4. Use SHAP or feature importance to interpret a supervised model that
   predicts a flood metric from the storm type.
